In [ ]:
import re
from pathlib import Path

from google.colab import drive

WHEEL_DIRECTORY = Path("/content/drive/MyDrive/data/jlens-reasoning/wheels")
REQUIREMENTS = WHEEL_DIRECTORY / "requirements-colab.txt"
COMMIT_FILE = WHEEL_DIRECTORY / "project-commit.txt"
DIRTY_FILE = WHEEL_DIRECTORY / "project-dirty.txt"

drive.mount("/content/drive")

if not COMMIT_FILE.is_file():
    raise RuntimeError(f"Missing project commit marker: {COMMIT_FILE}")
PROJECT_COMMIT = COMMIT_FILE.read_text(encoding="utf-8").strip()
if re.fullmatch(r"[0-9a-f]{40}", PROJECT_COMMIT) is None:
    raise RuntimeError("Project commit marker is invalid")
if not DIRTY_FILE.is_file():
    raise RuntimeError(f"Missing project dirty marker: {DIRTY_FILE}")
dirty_value = DIRTY_FILE.read_text(encoding="utf-8").strip()
if dirty_value not in {"true", "false"}:
    raise RuntimeError("Project dirty marker is invalid")
PROJECT_WORKING_TREE_DIRTY = dirty_value == "true"

wheels = sorted(WHEEL_DIRECTORY.glob("jlens_reasoning-*.whl"))
if not REQUIREMENTS.is_file():
    raise RuntimeError(f"Missing locked requirements: {REQUIREMENTS}")
if len(wheels) != 1:
    raise RuntimeError(
        f"Expected exactly one project wheel in {WHEEL_DIRECTORY}, found {len(wheels)}"
    )

wheel = wheels[0]
print(f"Installing locked environment from {REQUIREMENTS}")
%pip install -qq --disable-pip-version-check --requirement {REQUIREMENTS}
print(f"Installing project wheel {wheel.name}")
%pip install -qq --disable-pip-version-check --force-reinstall --no-deps {wheel}
print("Colab project installation complete")

del COMMIT_FILE, DIRTY_FILE, REQUIREMENTS, WHEEL_DIRECTORY, dirty_value, wheel, wheels

In [ ]:
from jlens_reasoning.environments.colab import initialize_colab

context = initialize_colab(enable_wandb=False, require_cuda=True)
context

In [ ]:
import jlens
import torch
import transformers
from datasets import load_from_disk

from jlens_reasoning.benchmarks.flenqa.runner import (
    ApplyLensRunner,
    LensRunners,
    RunConfig,
    run_benchmark,
)
from experiments.jlens_readout_sanity.constants import (
    LENS_PATH,
    LENS_REVISION,
    MODEL_NAME,
    MODEL_PATH,
)
from jlens_reasoning.benchmarks.flenqa.dataset import normalize_rows

dataset = load_from_disk(context.datasets_dir / "flenqa")
raw_rows = dataset["train"] if hasattr(dataset, "keys") else dataset
rows = normalize_rows(raw_rows, full=True)
causal_lm = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, local_files_only=True
).to(context.device)
tokenizer = transformers.AutoTokenizer.from_pretrained(
    MODEL_PATH, local_files_only=True
)
model = jlens.from_hf(causal_lm, tokenizer)
lens = jlens.JacobianLens.from_pretrained(LENS_PATH)
runners = LensRunners(
    ApplyLensRunner(lens, model, True), ApplyLensRunner(lens, model, False)
)

In [ ]:
len(rows)

In [ ]:
manifest = run_benchmark(
    rows,
    output_dir=context.runs_dir / "flenqa-full-run",
    tokenizer=tokenizer,
    runners=runners,
    config=RunConfig(
        model_name=MODEL_NAME,
        lens_revision=LENS_REVISION,
        tokenizer_name=MODEL_NAME,
        code_revision=PROJECT_COMMIT,
        expected_source_rows=12_000,
        expected_bridge_problems=200,
    ),
)
manifest